# Microzone Cleanup Notebook

This notebook takes a microzone and traffic analysis zone layer and fills in gaps, then merges them into neighbors without crossing over the taz boundry.

Point workspace variable at the `.gdb` folder for your project

Ensure that microzone and traffic analysis zone are inside of your `.gdb` folder

Point `maz` variable at the name of microzone feature class 

Example: `maz = "microzone_layer"`

Point `taz` variable at the name of traffic analysis zone feature class

Example: `taz = "taz_layer"`




### Import Necessary Libraries

In [2]:
import arcpy

### Assign key Variables

In [3]:
workspace = r"D:\ABM-INP-Microzones\Scripts\Scratch\Microzones_Cleaner\Microzones_Cleaner.gdb"
maz = "microzones_Export"
taz = "TAZ_Export"

### Point Arcpy at `.gdb` folder
Enable output overwrite

In [4]:
arcpy.env.workspace = workspace
arcpy.env.overwriteOutput = True

### Remove MAG TAZ
Since the TAZ layer inclues MAG TAZ's we can exclude them from our file since we only want to look at WFRC TAZ

In [5]:
temp_taz = "temp_taz"
wfrc_taz = "_01_wfrc_taz"


arcpy.management.MakeFeatureLayer(
    in_features=taz, 
    out_layer=temp_taz)

arcpy.management.SelectLayerByAttribute(
    in_layer_or_view=temp_taz, 
    selection_type="NEW_SELECTION", 
    where_clause="PLANAREA = 'WFRC' ")

arcpy.management.CopyFeatures(
    in_features=temp_taz,
    out_feature_class=wfrc_taz)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_01_wfrc_taz'>

### Run Identity Tool

This will fill in all gaps between polygons

Creates a new feature class called `maz_i` which should have more features than orignial maz becuase these are all new filled in featuers

Original Microzones total = 9,875

MAZID"0" = 861

New Zones = 869

MAZ identified  total = 10,744

In [6]:
maz_i = "_02_maz_i"

arcpy.analysis.Identity(
    in_features=wfrc_taz, 
    identity_features=maz, 
    out_feature_class=maz_i)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_02_maz_i'>

### Multipart to Singlepart

Identity will treat some new features a single one and this function will break them back into their own

Creates a new feature class called `maz_filled` with all gaps filled in as unique features

MAZ Filled total = 11,680

New Zones = 936

MAZID"0" = 1,793

NEW MAZID"0" = 924

Missing duplicate Zones = 12
The 12 duplicate zones will be filterd out later when I look for sliver zones


In [7]:
maz_filled = "_03_maz_filled"
arcpy.management.MultipartToSinglepart(
    in_features= maz_i,
    out_feature_class=maz_filled)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_03_maz_filled'>

### Create new field for filled gap polygon

All new polygons created from the identify and multipart functions will have `MAZID = 0` so to easily identify them later I will make a dummy column for it
Flags 1,793 polygons


In [8]:
arcpy.management.CalculateField(
    in_table=maz_filled,
    field="gap_flag",
    expression="1 if (!MAZID! == 0 or !MAZID! is None) else 0",
    expression_type="PYTHON3",
    field_type="SHORT"
)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_03_maz_filled'>

## Identify zones for review

In addition to gap zones we want to flag sliver zones

Shape Area is less than 10,000

AND 

PolsbyPopper is less than .03

OR 

PerimtertoArea is less than .03

OR

Shape Area < 1,000


**This will flag 1,986 zones as sliver zones to be merged into the neighboring microzones**

In [9]:
maz_sliver = "_04_maz_sliver"


arcpy.management.CopyFeatures(
    in_features= maz_filled,
    out_feature_class=maz_sliver
)

code_block = """
def check_sliver(pp, pa, area, gap):
    # Guard clause: large features (highways) are never slivers
    if area is not None and area >= 10000:
        return 0
    
    # Check sliver criteria for features under 10,000
    if pp is not None and pp < 0.03:
        return 1
    if pa is not None and pa < 0.03:
        return 1
    if area is not None and area < 1000:
        return 1
    if gap == 1:
        return 1
    return 0
"""

arcpy.management.CalculateField(
    in_table=maz_sliver,
    field="Sliver_Flag",
    expression="check_sliver(!PolsbyPopper!, !PerimtertoArea!, !Shape_Area!, !gap_flag!)",
    expression_type="PYTHON3",
    code_block=code_block,
    field_type="SHORT"
)


<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_04_maz_sliver'>

### Additional layer of just sliver zones for eye test

In [10]:
slivers = "_05_iso_slivers"

arcpy.management.MakeFeatureLayer(
    in_features=maz_sliver, 
    out_layer=slivers
)

arcpy.management.SelectLayerByAttribute(
    in_layer_or_view=slivers, 
    selection_type="NEW_SELECTION", 
    where_clause="Sliver_Flag = 1"
)

arcpy.management.CopyFeatures(
    in_features=slivers,
    out_feature_class=slivers
)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_05_iso_slivers'>

### Create Taz vector lines

Eliminate tool requires vector lines to use as a boarder so we dont merge between TAZ

In [11]:
taz_lines = "_06_taz_lines"
arcpy.management.PolygonToLine(taz, taz_lines)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_06_taz_lines'>

### Run Eliminate 

Merges 1,810 features which leaves 176 unmerged features

These are zones that have micro crossover TAZ boundaries



In [12]:
maz_cleaned = "_07_maz_cleaned"
elim_layer = "temp_elim_layer"

arcpy.management.MakeFeatureLayer(
    in_features=maz_sliver, 
    out_layer=elim_layer
)

arcpy.management.SelectLayerByAttribute(elim_layer, "NEW_SELECTION", 'Sliver_Flag = 1')

arcpy.management.Eliminate(
    in_features=elim_layer, 
    out_feature_class=maz_cleaned,
    ex_features=taz_lines
)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_07_maz_cleaned'>

### Snap micro zones to the taz lines to get rid of micro cross over

These zones are all small enough to be automatically deleted with Integrate tool

In [13]:
maz_snapped = "_08_maz_snapped"

arcpy.management.CopyFeatures(maz_cleaned, maz_snapped)
arcpy.management.Integrate(
    in_features=[[maz_snapped, 1]], 
    cluster_tolerance="1 Meters" # Adjust to "3 Feet" if your projection is in feet
)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_08_maz_snapped'>

Double check to see if there is any microzones that overlap a taz

In [15]:
snapped_layer = "temp_snapped_layer"
maz_crossover = "_09_maz_crossover"

arcpy.management.MakeFeatureLayer(maz_snapped, snapped_layer)

arcpy.management.SelectLayerByLocation(
    in_layer=snapped_layer,
    overlap_type="CROSSED_BY_THE_OUTLINE_OF",
    select_features=taz
)

arcpy.management.CopyFeatures(snapped_layer, maz_crossover)



<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_09_maz_crossover'>

In [17]:
maz_shrunk = "_10_maz_shrunk"
maz_crossover_lines = "_11_maz_crossover_lines"

# 1. Shrink the microzones slightly to detach perfectly shared borders
arcpy.analysis.Buffer(
    in_features=maz_snapped, 
    out_feature_class=maz_shrunk, 
    buffer_distance_or_field="-0.1 Meters" # Change to "-0.5 Feet" if your map is in feet
)

# 2. Clip the TAZ boundaries using the shrunk microzones
arcpy.analysis.Clip(
    in_features=taz_lines,
    clip_features=maz_shrunk,
    out_feature_class=maz_crossover_lines
)

<Result 'D:\\ABM-INP-Microzones\\Scripts\\Scratch\\Microzones_Cleaner\\Microzones_Cleaner.gdb\\_11_maz_crossover_lines'>

summary stats for maz per taz, county, city 

count

min

max

avg



be able to add in the MAG microzones